# 🚀 Space Biology RAG System

**Retrieval-Augmented Generation for NASA Space Biology Research**

This notebook builds a semantic search system to query 607 scientific articles about microgravity effects on biological systems.

## System Overview
Documents (607) → Chunks (13,453) → Embeddings (768-dim) → Vector Index → Query → Answer

### 📝 Summary
 
**What we built:**
- ✅ Loaded 607 scientific articles
- ✅ Created 13,453 overlapping chunks
- ✅ Generated 768-dim semantic embeddings
- ✅ Built fast vector search index
- ✅ Implemented AI-powered Q&A
 
**Performance:**
- Index build time: ~10 minutes (one-time)
- Query time: <1 second
- Memory usage: ~3 GB

In [ ]:
import json
import logging
import os
from typing import List, Dict, Any, Tuple
from dataclasses import dataclass

from sentence_transformers import SentenceTransformer
from annoy import AnnoyIndex
from transformers import pipeline

# Setup logging
logging.basicConfig(level=logging.INFO, format='%(levelname)s: %(message)s')
logger = logging.getLogger(__name__)

In [ ]:
@dataclass
class Config:
    """System configuration"""
    input_json_file: str = 'resources/space_biology_scraped.json'
    index_file: str = 'space_biology_index.ann'
    embedding_model: str = 'paraphrase-multilingual-mpnet-base-v2'
    qa_model: str = 'google/flan-t5-base'
    embedding_size: int = 768
    chunk_max_size: int = 300
    chunk_overlap: int = 50
    annoy_trees: int = 10
    top_k_results: int = 5

config = Config()
print(f"✓ Configuration loaded")
print(f"  - Chunk size: {config.chunk_max_size} tokens")
print(f"  - Overlap: {config.chunk_overlap} tokens")
print(f"  - Index file: {config.index_file}")

✓ Configuration loaded
  - Chunk size: 300 tokens
  - Overlap: 50 tokens
  - Index file: space_biology_index.ann


In [ ]:
def load_documents(filepath: str) -> List[Dict[str, Any]]:
    """Load scientific articles from JSON"""
    try:
        with open(filepath, 'r', encoding='utf-8') as f:
            docs = json.load(f)
        logger.info(f"✓ Loaded {len(docs)} documents")
        return docs
    except Exception as e:
        logger.error(f"Error loading documents: {e}")
        raise

# Load the data
documents = load_documents(config.input_json_file)
print(f"\n📊 Dataset: {len(documents)} articles")

INFO: ✓ Loaded 607 documents



📊 Dataset: 607 articles


In [ ]:
def chunk_text(text: str, max_size: int = 300, overlap: int = 10) -> List[str]:
    """Split text into overlapping chunks"""
    tokens = text.split()
    chunks = []
    i = 0
    while i < len(tokens):
        chunks.append(" ".join(tokens[i:i+max_size]))
        i += max_size - overlap
    return chunks

# Process documents into chunks
chunked_documents = []
chunk_id = 0

for doc_id, doc in enumerate(documents):
    # Combine all sections
    full_text = ""
    if 'title' in doc:
        full_text += doc['title'] + " "
    if 'abstract' in doc:
        full_text += doc['abstract'] + " "
    if 'sections' in doc:
        for content in doc['sections'].values():
            if isinstance(content, str):
                full_text += content + " "
    
    # Create chunks
    chunks = chunk_text(full_text, max_size=300, overlap=10)
    
    for chunk_text_str in chunks:
        chunked_documents.append({
            "id": chunk_id,
            "document_id": doc_id,
            "pmc_id": doc.get('pmc_id', 'N/A'),
            "title": doc.get('title', 'N/A'),
            "text": chunk_text_str
        })
        chunk_id += 1

print(f"Total chunks: {len(chunked_documents)}")

Total chunks: 13453


In [ ]:
print("Loading embedding model...")
model = SentenceTransformer(config.embedding_model)

# Generate embeddings in batches (MUCH faster)
print(f"Generating embeddings for {len(chunked_documents)} chunks...")
texts = [doc["text"] for doc in chunked_documents]

# Batch encode (3-5 minutes for 15k chunks)
embeddings = model.encode(texts, batch_size=32, show_progress_bar=True)

# Assign to documents
for i, embedding in enumerate(embeddings):
    chunked_documents[i]["embedding"] = embedding

print("Embeddings complete")

In [ ]:
chunk_mapping = {doc["id"]: doc for doc in chunked_documents}

if os.path.exists(config.index_file):
    print(f"Loading index from {config.index_file}...")
    index = AnnoyIndex(config.embedding_size, 'angular')
    index.load(config.index_file)
    print("✓ Index loaded")
else:
    print("Building index...")
    index = AnnoyIndex(config.embedding_size, 'angular')
    
    for doc in chunked_documents:
        index.add_item(doc["id"], doc["embedding"])
    
    index.build(config.annoy_trees)
    index.save(config.index_file)
    print(f"✓ Index saved to {config.index_file}")

Building index...
✓ Index saved to space_biology_index.ann


In [ ]:
print("Loading QA model...")
qa_generator = pipeline('text2text-generation', model='google/flan-t5-small')
print("QA model loaded")

def generate_answer(question: str, context_chunks: List[Dict]) -> str:
    """Generate answer from context"""
    context = "\n---\n".join([c["text"] for c in context_chunks])[:1500]
    
    prompt = f"""Context: {context}

Question: {question}

Answer:"""
    
    result = qa_generator(prompt, max_length=100, num_beams=1, do_sample=False)
    return result[0]['generated_text']

Loading QA model...


Device set to use mps:0


QA model loaded


In [ ]:
def query(question: str, top_k: int = 5):
    """Query the RAG system"""
    # Get query embedding (fast)
    query_embedding = model.encode(question)
    
    # Search for similar chunks (fast)
    chunk_ids = index.get_nns_by_vector(query_embedding, top_k)
    relevant_chunks = [chunk_mapping[idx] for idx in chunk_ids]
    
    # Generate answer (takes ~5-10 seconds)
    answer = generate_answer(question, relevant_chunks)
    
    return answer, relevant_chunks

def print_results(question: str, answer: str, chunks: List[Dict]):
    """Print results"""
    print("\n" + "="*80)
    print(f"Q: {question}")
    print("="*80)
    print(f"A: {answer}")
    
    unique_docs = list(set([documents[c["document_id"]]["title"] for c in chunks]))
    
    print(f"\nSources ({len(unique_docs)}):")
    for i, title in enumerate(unique_docs, 1):
        print(f"  {i}. {title}")
    
    print(f"\nChunks ({len(chunks)}):")
    for i, chunk in enumerate(chunks, 1):
        preview = chunk["text"][:100].replace("\n", " ")
        print(f"  {i}. {preview}...")

In [ ]:
question = "What is microgravity?"
answer, chunks = query(question)
print_results(question, answer, chunks)

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Both `max_new_tokens` (=256) and `max_length`(=100) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



Q: What is microgravity?
A: microgravity

Sources (3):
  1. RNAseq Analysis of the Response ofArabidopsis thalianato Fractional Gravity Under Blue-Light Stimulation During Spaceflight
  2. Physical Forces Modulate Oxidative Status and Stress Defense Meditated Metabolic Adaptation of Yeast Colonies: Spaceflight and Microgravity Simulations
  3. Microgravity and Cellular Biology: Insights into Cellular Responses and Implications for Human Health

Chunks (5):
  1. Microgravity and Cellular Biology: Insights into Cellular Responses and Implications for Human Healt...
  2. positioning. Defining the utility of microgravity and microgravity simulations is confounded by use ...
  3. this state [ 1 , 2 ]. In cellular biology, microgravity presents a unique opportunity to study how c...
  4. of microgravity ( Vandenbrink et al., 2016 ). This relationship was shown to be linearly related to ...
  5. ]. Microgravity presents a unique and multifaceted challenge for human biology, profoundly influe

In [ ]:
question = "How does microgravity affect the immune system?"
answer, chunks = query(question)
print_results(question, answer, chunks)

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Both `max_new_tokens` (=256) and `max_length`(=100) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



Q: How does microgravity affect the immune system?
A: radiation, isolation, and short-term exposure to hypergravity

Sources (3):
  1. Impact of microgravity and lunar gravity on murine skeletal and immune systems during space travel
  2. Microgravity and Cellular Biology: Insights into Cellular Responses and Implications for Human Health
  3. Spaceflight and simulated microgravity conditions increase virulence ofSerratia marcescensin theDrosophila melanogasterinfection model

Chunks (5):
  1. microgravity, but also radiation, isolation, and short-term exposure to hypergravity. Previous studi...
  2. microgravity, but also radiation, isolation, and short-term exposure to hypergravity. Previous studi...
  3. Microgravity and Cellular Biology: Insights into Cellular Responses and Implications for Human Healt...
  4. ]. Microgravity presents a unique and multifaceted challenge for human biology, profoundly influenci...
  5. Impact of microgravity and lunar gravity on murine skeletal and 

In [ ]:
question = "What happens to the liver in space?"
answer, chunks = query(question)
print_results(question, answer, chunks)

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Both `max_new_tokens` (=256) and `max_length`(=100) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



Q: What happens to the liver in space?
A: a marked reduction of diuresis and natriuresis

Sources (5):
  1. Spaceflight Activates Autophagy Programs and the Proteasome in Mouse Liver
  2. Cosmic kidney disease: an integrated pan-omic, physiological and morphological study into spaceflight-induced renal dysfunction
  3. Multi-omics analysis of multiple missions to space reveal a theme of lipid dysregulation in mouse liver
  4. Analyzing the relationship between gene expression and phenotype in space-flown mice using a causal inference machine learning ensemble
  5. Muscle atrophy phenotype gene expression during spaceflight is linked to a metabolic crosstalk in both the liver and the muscle in mice

Chunks (5):
  1. as a few weeks into a mission. Notably, the effects of LEO spaceflight on many other organ systems a...
  2. mouse liver. Furthermore, our multi-‘omics studies suggest that accumulation of oxidized proteins co...
  3. Multi-omics analysis of multiple missions to space revea

In [ ]:
question = "How do stem cells behave in microgravity?"
answer, chunks = query(question)
print_results(question, answer, chunks)

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Both `max_new_tokens` (=256) and `max_length`(=100) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



Q: How do stem cells behave in microgravity?
A: cytoskeletal organization, cell adhesion, and intracellular signaling

Sources (4):
  1. Discoveries from human stem cell research in space that are relevant to advancing cellular therapies on Earth
  2. Selective Proliferation of Highly Functional Adipose-Derived Stem Cells in Microgravity Culture with Stirred Microspheres
  3. THE INDIVIDUAL AND COMBINED EFFECTS OF SPACEFLIGHT RADIATION AND MICROGRAVITY ON BIOLOGIC SYSTEMS AND FUNCTIONAL OUTCOMES
  4. Microgravity and Cellular Biology: Insights into Cellular Responses and Implications for Human Health

Chunks (5):
  1. this state [ 1 , 2 ]. In cellular biology, microgravity presents a unique opportunity to study how c...
  2. to long-duration spaceflight [ 24 ]. Simulated microgravity significantly enhances the differentiati...
  3. for the use of micro-physiological systems and organ-based cultures in space. These programs, and fo...
  4. Selective Proliferation of Highly Functional A

In [ ]:
question = "Do bacteria become more dangerous in space?"
answer, chunks = query(question)
print_results(question, answer, chunks)

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Both `max_new_tokens` (=256) and `max_length`(=100) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



Q: Do bacteria become more dangerous in space?
A: i.e., the host immune system may be responding differently to spaceflight-altered bacteria.

Sources (4):
  1. Characterization of the total and viable bacterial and fungal communities associated with the International Space Station surfaces
  2. Effect of spaceflight onPseudomonas aeruginosafinal cell density is modulated by nutrient and oxygen availability
  3. Spaceflight Promotes Biofilm Formation byPseudomonas aeruginosa
  4. Spaceflight and simulated microgravity conditions increase virulence ofSerratia marcescensin theDrosophila melanogasterinfection model

Chunks (5):
  1. consistent with previous studies, which found that spaceflight-induced changes to pathogens, such as...
  2. consistent with previous studies, which found that spaceflight-induced changes to pathogens, such as...
  3. associated with space flight [ 19 , 20 ] and the lack of sophisticated medical interventions that ar...
  4. aspects of the spaceflight environ